In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


panel_data_path = config.PROJECT_ROOT/ "data" / "panel_data.csv"
panel_data = pd.read_csv(panel_data_path)

Train/test split must be time-aware. Entries in the train set should all predate entries in the test set.

In [2]:
year_counts = panel_data.groupby('Year').size().reset_index(name = 'NumberOfRows')
year_counts = year_counts.sort_values('Year')
year_counts['Cumulative'] = year_counts['NumberOfRows'].cumsum()
total_rows = year_counts.loc[len(year_counts)-1, 'Cumulative']
year_counts['EntriesPrecedingAndIncluding(%)'] = (year_counts['Cumulative']/total_rows) *100
year_counts

,Year,NumberOfRows,Cumulative,EntriesPrecedingAndIncluding(%)
0,2015,20521,20521,6.723501
1,2016,26591,47112,15.435778
2,2017,31939,79051,25.900273
3,2018,37641,116692,38.232972
4,2019,40916,157608,51.638692
5,2020,17673,175281,57.429074
6,2021,25873,201154,65.906105
7,2022,27153,228307,74.802515
8,2023,35947,264254,86.580192
9,2024,40959,305213,100.000000


In [3]:
numbers = panel_data.groupby('CompetesNextYear').size().reset_index(name = 'Number')
proportions = numbers.copy()
proportions['Proportion'] = numbers['Number']/numbers['Number'].sum()
proportions

,CompetesNextYear,Number,Proportion
0,False,180574,0.591633
1,True,124639,0.408367


In [4]:
panel_data['CovidAffectedPrediction'] = (
    panel_data['Year'].isin([2019, 2020, 2021])
).astype(int)

panel = panel_data.drop(columns = ['Name', 'WeightClassKg', 'Sex'])
train = panel[panel['Year']<=2022].copy()
validate = panel[panel['Year']==2023].copy()
test = panel[panel['Year']>2023].copy()


train_X = train.drop(columns = 'CompetesNextYear')
train_y = train['CompetesNextYear']
test_X = test.drop(columns = 'CompetesNextYear')
test_y = test['CompetesNextYear']

clf = RandomForestClassifier(random_state=42)
clf.fit(train_X, train_y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [5]:
preds = clf.predict(test_X)
acc = accuracy_score(test_y,preds)
precision = precision_score(test_y, preds)
recall = recall_score(test_y, preds)
f1 = f1_score(test_y, preds)

print(acc)
print(f1)
print(precision)
print(recall)

0.6425693986669596
0.6278029185946001
0.6278667683702008
0.6277390818038537


Federation proportion improves performance of random forest but decreases performance of gradient bossted

In [6]:
clf2 = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf2.fit(train_X, train_y)

preds2 = clf2.predict(test_X)
acc2 = accuracy_score(test_y,preds2)
precision2 = precision_score(test_y, preds2)
recall2 = recall_score(test_y, preds2)
f12 = f1_score(test_y, preds2)

print(acc2)
print(precision2)
print(recall2)
print(f12)

0.6549720452159477
0.6410700636942676
0.63963597539275
0.640352216623403


In [7]:
importances = clf.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': train_X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False, ignore_index =True)

feature_importance_df

,Feature,Importance
0,TimeSinceLastPBYearEnd,7.491840e-02
1,Age,6.897093e-02
2,Wilks,6.331036e-02
3,Goodlift,6.295135e-02
4,Dots,6.281319e-02
5,TimeCompetingYearEnd,6.170075e-02
6,BestMeetOfYear,5.920060e-02
7,FederationProportion,5.822325e-02
8,PercentageImprovementGradientWithinYear,4.769442e-02
9,ImprovementGradientWithinYear,4.764839e-02


In [9]:
#reduced_feats = panel_data.drop(columns = ['Goodlift', 'Dots', 'ImprovementGradientWithinYear','ImprovementGradientBetweenYears', 'ImprovementAcceleration'])
important_feats = feature_importance_df.loc[feature_importance_df['Importance']>0.001]

features = important_feats['Feature'].to_list()
columns = features + ['CompetesNextYear']


train_red = train[columns]
train_red_X = train_red.drop(columns = 'CompetesNextYear')
train_red_y = train_red['CompetesNextYear']

test_red = validate[columns]
test_red_X = test_red.drop(columns = 'CompetesNextYear')
test_red_y = test_red['CompetesNextYear']

In [10]:
clf3 = RandomForestClassifier(random_state=42)
clf3.fit(train_red_X, train_red_y)

preds3 = clf3.predict(test_red_X)
acc3 = accuracy_score(test_red_y,preds3)
precision3 = precision_score(test_red_y, preds3)
recall3 = recall_score(test_red_y, preds3)
f13 = f1_score(test_red_y, preds3)

In [11]:
print(acc3)
print(precision3)
print(recall3)
print(f13)

0.6484824881074915
0.6390048814915015
0.6257559177561481
0.6323110050631437


In [12]:
clf4 = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf4.fit(train_red_X, train_red_y)


preds4 = clf4.predict(test_red_X)
acc4 = accuracy_score(test_red_y,preds4)
precision4 = precision_score(test_red_y, preds4)
recall4 = recall_score(test_red_y, preds4)
f14 = f1_score(test_red_y, preds4)

In [13]:
print(acc4)
print(precision4)
print(recall4)
print(f14)

0.658525050769188
0.6499116087212728
0.6352012900996371
0.642472257012204


In [15]:
print(features)

['TimeSinceLastPBYearEnd', 'Age', 'Wilks', 'Goodlift', 'Dots', 'TimeCompetingYearEnd', 'BestMeetOfYear', 'FederationProportion', 'PercentageImprovementGradientWithinYear', 'ImprovementGradientWithinYear', 'Year', 'WeightClassKgCode', 'ImprovementGradientTwoMeets', 'PercentageImprovementGradientTwoMeets', 'FederationRankCapped', 'NumberOfMeets', 'PercentageImprovementGradientBetweenYears', 'ImprovementGradientBetweenYears', 'CovidAffectedPrediction', 'ImprovementAcceleration', 'ImprovementAccelerationForPercentage', 'AverageAttemptsMade', 'LeastAttemptsMade', 'LastMeetAttemptsMade', 'MostAttemptsMade', 'F', 'M', 'PercentageBombOuts', 'IPF', 'EPF', 'MeetsSinceLastBombOut', 'NumberOfBombOuts']


In [ ]:
d = {'acc':[acc, acc2, acc3, acc4], 'precision': [precision, precision1, precision2, precision3, precision4]
    'recall': [recall, recall2, recall3, recall4], 'f1': [f1, f12, f13, f14]}
results = pd.DataFrame(data =d)

# How many features to keep

Will take top N features where N is determined by taking top n features and seeing which value of n yields best performance on validation set. Can then use top N features to retrain classifier and evaluate performance using test set.

In [11]:
importances = clf.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': train_X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False, ignore_index =True)

feature_importance_df

,Feature,Importance
0,TimeSinceLastPBYearEnd,7.469893e-02
1,Age,6.874089e-02
2,Wilks,6.374427e-02
3,Goodlift,6.332242e-02
4,Dots,6.210444e-02
5,TimeCompetingYearEnd,6.193519e-02
6,FederationProportion,5.924999e-02
7,BestMeetOfYear,5.913816e-02
8,PercentageImprovementGradientWithinYear,4.766649e-02
9,ImprovementGradientWithinYear,4.755435e-02


In [12]:
panel[['Wilks','Dots','Goodlift']].corr()


,Wilks,Dots,Goodlift
Wilks,1.000000,0.997941,0.992439
Dots,0.997941,1.000000,0.996222
Goodlift,0.992439,0.996222,1.000000


Can also see that percentage-based improvement features perform better than raw improvement. Will drop raw improvement features

In [ ]:
reduced_feats = panel_data.drop(columns = ['Goodlift', 'Dots', 'ImprovementGradientWithinYear','ImprovementGradientBetweenYears', 'ImprovementAcceleration'])
reduced_feats = reduced_feats.loc[reduced_feats['Importance']>0.001]

Wilks, Goodlift and Dots are all different formulas for bodyweight adjusted strength. Very high correlation between these so will only keep most predictive feature, found to be Dots.

In [26]:
# reduced_feats = feature_importance_df.loc[:13]
# reduced_feats = reduced_feats.drop(labels = [2,4], axis = 0)
# reduced_feats[:4]

,Feature,Importance
0,TimeSinceLastPBYearEnd,0.074699
1,Age,0.068741
3,Goodlift,0.063322
5,TimeCompetingYearEnd,0.061935


In [15]:
results = []

for i in range(len(feature_importance_df)):
    features = feature_importance_df.loc[:i, 'Feature'].to_list()
    columns = features + ['CompetesNextYear']

    train_n = train[columns]
    train_n_X = train_n.drop(columns = 'CompetesNextYear')
    train_n_y = train_n['CompetesNextYear']

    validate_n = validate[columns]
    validate_n_X = validate_n.drop(columns = 'CompetesNextYear')
    validate_n_y = validate_n['CompetesNextYear']

    clf_n = RandomForestClassifier(random_state=42)
    clf_n.fit(train_n_X, train_n_y)
    
    preds_n = clf_n.predict(validate_n_X)
    acc_n = accuracy_score(validate_n_y, preds_n)
    f1_n = f1_score(validate_n_y, preds_n)
    precision_n = precision_score(validate_n_y, preds_n)
    recall_n = recall_score(validate_n_y, preds_n)

    results.append({'n_features': len(features), 'features': features, 
                    'accuracy': acc_n,'f1': f1_n, 'precision_n': precision_n, 'recall_n':recall_n})

results_df = pd.DataFrame(results)

KeyboardInterrupt: 

In [ ]:
results_df